#Imports

In [ ]:
!pip install woodelf_explainer==0.2.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [ ]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Environment Note
All the code below, that works with the EEEI-CIS fraud data will run perfectly in the free version of Google Colab.
The free version machine have 12GB RAM and this is enough for this part of the notebook.

Next, we will load the KDD-Cup dataset. As it includes millions of rows we need more than 12GB to run the algorithm.
This part of the notebook needs the 50GB CPU machine. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# Fraud Data Preprocessing and Model training

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/ShapResearch/HighDepth/IJCAI/data/ieee_cis_fraud_train.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/ShapResearch/HighDepth/IJCAI/data/ieee_cis_fraud_test.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [ ]:
gbm_100trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18,21])




Trained on depth 6
	Paths of depth 1: 6 paths
	Paths of depth 2: 60 paths
	Paths of depth 3: 135 paths
	Paths of depth 4: 284 paths
	Paths of depth 5: 473 paths
	Paths of depth 6: 2086 paths



Trained on depth 9
	Paths of depth 1: 16 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 138 paths
	Paths of depth 4: 258 paths
	Paths of depth 5: 454 paths
	Paths of depth 6: 710 paths
	Paths of depth 7: 959 paths
	Paths of depth 8: 1222 paths
	Paths of depth 9: 3752 paths



Trained on depth 12
	Paths of depth 1: 15 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 143 paths
	Paths of depth 4: 290 paths
	Paths of depth 5: 472 paths
	Paths of depth 6: 636 paths
	Paths of depth 7: 884 paths
	Paths of depth 8: 1140 paths
	Paths of depth 9: 1496 paths
	Paths of depth 10: 1681 paths
	Paths of depth 11: 1991 paths
	Paths of depth 12: 4782 paths



Trained on depth 15
	Paths of depth 1: 16 paths
	Paths of depth 2: 72 paths
	Paths of depth 3: 121 paths
	Paths of depth 4: 278 paths
	Paths of

In [ ]:
gbm_10trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=10, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 6 paths
	Paths of depth 4: 54 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 89 paths
	Paths of depth 7: 137 paths
	Paths of depth 8: 158 paths
	Paths of depth 9: 191 paths
	Paths of depth 10: 204 paths
	Paths of depth 11: 248 paths
	Paths of depth 12: 552 paths



Trained on depth 15
	Paths of depth 2: 1 paths
	Paths of depth 3: 5 paths
	Paths of depth 4: 52 paths
	Paths of depth 5: 61 paths
	Paths of depth 6: 85 paths
	Paths of depth 7: 158 paths
	Paths of depth 8: 141 paths
	Paths of depth 9: 192 paths
	Paths of depth 10: 215 paths
	Paths of depth 11: 269 paths
	Paths of depth 12: 265 paths
	Paths of depth 13: 290 paths
	Paths of depth 14: 319 paths
	Paths of depth 15: 642 paths



Trained on depth 18
	Paths of depth 2: 1 paths
	Paths of depth 3: 7 paths
	Paths of depth 4: 50 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 74 paths
	Paths of depth 7: 151 paths
	Paths of depth 8: 161 paths
	Paths of depth 9: 176 paths
	Paths of dep

In [ ]:
gbm_1trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=1, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 30 paths



Trained on depth 15
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 30 paths



Trained on depth 18
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 11 paths
	Paths 

In [ ]:
def pd_high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.high_depth_woodelf.woodelf_for_high_depth(model, consumer_data, background_data=None, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def pd_simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_path_dependent_metric(model, consumer_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def pd_shap_on_models_dict(models, consumer_data: pd.DataFrame, interaction_values: bool = False):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
        if interaction_values:
            simple_shap_values = explainer.shap_interaction_values(consumer_data)
        else:
            simple_shap_values = explainer.shap_values(consumer_data)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Path Dependent SHAP

In [ ]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18], 21: gbm_10trees[21]},
    consumer_data=transactions_test[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Path Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:06<00:00, 15.24it/s]


M time: 0.0 sec, s time: 0.39 sec (f prepare time: 0.1314249038696289)
On Depth 6 Took: 6.676840782165527


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:23<00:00,  4.21it/s]


M time: 0.01 sec, s time: 1.62 sec (f prepare time: 0.4812285900115967)
On Depth 9 Took: 24.007874011993408


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:58<00:00,  1.71it/s]


...

M time: 1.18 sec, s time: 55.51 sec (f prepare time: 5.386950492858887)
On Depth 15 Took: 184.38063883781433


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [14:22<00:00,  8.62s/it]


M time: 11.28 sec, s time: 573.62 sec (f prepare time: 32.02187705039978)
On Depth 18 Took: 874.8558313846588


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [16:40<00:00, 100.06s/it]

M time: 104.26 sec, s time: 814.75 sec (f prepare time: 46.32819628715515)
On Depth 21 Took: 1105.0850186347961
Background SHAP: 6.676840782165527 & 24.007874011993408 & 59.01933169364929 & 184.38063883781433 & 874.8558313846588 & 1105.0850186347961


In [ ]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM
    consumer_data=transactions_test[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Path Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 268.56it/s]


cache misses: 71, cache used: 2973


Computing the values: 100%|██████████| 100/100 [00:05<00:00, 19.35it/s]


On Depth 6 Took: 5.6622092723846436


Preprocessing the trees: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]


...

On Depth 12 Took: 333.76311564445496


Preprocessing the trees: 100%|██████████| 1/1 [20:12<00:00, 1212.39s/it]


cache misses: 29, cache used: 112


Computing the values: 100%|██████████| 1/1 [00:02<00:00,  2.23s/it]

On Depth 15 Took: 1214.789384841919
Background SHAP: 5.6622092723846436 & 59.24622058868408 & 333.76311564445496 & 1214.789384841919


In [ ]:
shap_running_times = pd_shap_on_models_dict(
    gbm_100trees, consumer_data=fraud_test, interaction_values=False
)
print("Path Dependent SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 6 Took: 20.01420783996582


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 9 Took: 81.67084574699402


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 12 Took: 214.814439535141


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


On Depth 15 Took: 422.86936116218567
On Depth 18 Took: 707.5852506160736
Background SHAP: 20.01420783996582 & 81.67084574699402 & 214.814439535141 & 422.86936116218567 & 707.5852506160736


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


# Path Depedent SHAP IV

In [ ]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_10trees[18]}, # depth 21 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Path Dependent SHAP: " + " & ".join([str(round(woodelf_running_times[d])) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:02<00:00, 38.13it/s]


M time: 0.0 sec, s time: 0.43 sec (f prepare time: 0.1259617805480957)
On Depth 6 Took: 2.74495005607605


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:12<00:00,  8.12it/s]


M time: 0.02 sec, s time: 2.25 sec (f prepare time: 0.47887253761291504)
On Depth 9 Took: 12.621885061264038


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:58<00:00,  1.71it/s]


M time: 0.32 sec, s time: 22.66 sec (f prepare time: 1.551630973815918)
On Depth 12 Took: 59.36636018753052


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [08:05<00:00,  4.86s/it]


M time: 5.18 sec, s time: 370.58 sec (f prepare time: 5.569476366043091)
On Depth 15 Took: 491.7946877479553


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [14:39<00:00, 87.94s/it]

M time: 60.47 sec, s time: 778.38 sec (f prepare time: 5.479360580444336)
On Depth 18 Took: 939.978508234024
Background SHAP: 2.74495005607605 & 12.621885061264038 & 59.36636018753052 & 491.7946877479553 & 939.978508234024


In [ ]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12]}, # depth 15 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Path Dependent SHAP: " + " & ".join([str(round(woodelf_running_times[d])) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 159.67it/s]


cache misses: 71, cache used: 2973


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 62.06it/s]


On Depth 6 Took: 2.363471031188965


Preprocessing the trees: 100%|██████████| 100/100 [01:13<00:00,  1.36it/s]


...

On Depth 9 Took: 85.44551682472229


Preprocessing the trees: 100%|██████████| 10/10 [14:13<00:00, 85.37s/it]


cache misses: 179, cache used: 1523


Computing the values: 100%|██████████| 10/10 [00:14<00:00,  1.49s/it]

On Depth 12 Took: 868.6793177127838
Background SHAP: 2.363471031188965 & 85.44551682472229 & 868.6793177127838


In [ ]:
shap_running_times = pd_shap_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_1trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]},
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), interaction_values=True
)
print("Path Dependent SHAP: " + " & ".join([str(round(shap_running_times[d])) for d in sorted(list(shap_running_times.keys()))]))

On Depth 6 Took: 223.09566164016724
On Depth 9 Took: 1787.3755054473877
On Depth 12 Took: 1010.5316052436829
On Depth 15 Took: 116.70839476585388
On Depth 18 Took: 246.32422399520874
On Depth 21 Took: 523.36505818367
Path Dependent SHAP: 223.09566164016724 & 1787.3755054473877 & 1010.5316052436829 & 116.70839476585388 & 246.32422399520874 & 523.36505818367


# RAM Crashes

In [ ]:
# Crashed due to RAM

woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {18: gbm_1trees[18]},
    consumer_data=transactions_test[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM

woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {15: gbm_1trees[15]},
    consumer_data=transactions_test[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM (in the HighDepthPathToMatrices.build_matrices phase)

woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {21: gbm_1trees[21]},
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))